In [ ]:
from typing import TypedDict

class AssistantState(TypedDict):
    message: str
    intent: str
    risk: str
    plan: str
    draft: str
    decision: str
    critic_count: int
    final_answer: str

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

responder_llm = ChatOpenAI(
    model="deepseek/deepseek-v3.2",
    temperature=0.5,
    api_key='',
    base_url="https://openrouter.ai/api/v1"
)

critic_llm = ChatOpenAI(
    model="openai/gpt-4.1-mini",
    temperature=0.1,
    api_key='',
    base_url="https://openrouter.ai/api/v1"
)


In [ ]:
def detect_intent_and_risk(state: AssistantState):
    prompt = f"""
پیام کاربر:
{state["message"]}

۱. هدف پیام را مشخص کن
۲. سطح حساسیت را فقط با یکی از این کلمات تعیین کن:
low
medium
high

خروجی را دقیقاً در این قالب بده:
intent: ...
risk: ...
"""
    result = critic_llm.invoke(prompt).content
    lines = result.splitlines()
    state["intent"] = lines[0].replace("intent:", "").strip()
    state["risk"] = lines[1].replace("risk:", "").strip()
    return state

In [ ]:
def planner(state: AssistantState):
    prompt = f"""
کاربر فارسی زبان است.
بر اساس هدف زیر یک برنامه کوتاه برای پاسخ بنویس:

هدف:
{state["intent"]}

سطح حساسیت:
{state["risk"]}
"""
    state["plan"] = responder_llm.invoke(prompt).content
    return state

In [ ]:
def responder(state: AssistantState):
    prompt = f"""
بر اساس این برنامه پاسخ اولیه تولید کن:

برنامه:
{state["plan"]}

پیام کاربر:
{state["message"]}
"""
    state["draft"] = responder_llm.invoke(prompt).content
    return state

In [ ]:
def critic(state: AssistantState):

    n = state.get("critic_count", 0)

    if n >= 5:
        state["decision"] = "approve"
        return state

    prompt = f"""
تو یک منتقد کیفیت پاسخ هستی.

درخواست کاربر:

{state["message"]}

پاسخ تولید شده:

{state["draft"]}

این {n} امین بار است که این پاسخ را بررسی می‌کنی.

بررسی کن:

1. آیا پاسخ واقعاً به درخواست کاربر جواب داده؟
2. آیا اطلاعات ناقص یا اشتباه دارد؟
3. آیا نیاز به اصلاح جدی دارد؟

قانون اضمحلال:

- n=0 بسیار سخت‌گیر باش.
- n=1 و n=2 سخت‌گیری متوسط.
- n=3 فقط ایرادهای مهم را رد کن.
- n=4 فقط ایرادهای خیلی جدی را رد کن.
- n>=5 حتما approve.

فقط یکی از این دو کلمه را برگردان:

approve

یا

revise
"""

    decision = critic_llm.invoke(prompt).content.strip().lower()

    state["critic_count"] = n + 1
    state["decision"] = decision

    return state


In [ ]:
def improve(state: AssistantState):

    n = state.get("critic_count", 0)

    prompt = f"""
این {n} امین بار است که این پاسخ برای اصلاح به تو داده می‌شود.

اگر اصلاحات قبلی کافی بوده‌اند فقط بهبودهای جزئی اعمال کن.
اگر n به 5 نزدیک است از بازنویسی‌های سنگین خودداری کن.

پاسخ فعلی:

{state["draft"]}

نسخه بهبود یافته را برگردان.
"""

    state["draft"] = responder_llm.invoke(prompt).content
    return state


In [ ]:
def finalize(state: AssistantState):
    state["final_answer"] = state["draft"]
    return state

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AssistantState)

graph.add_node("detector", detect_intent_and_risk)
graph.add_node("planner", planner)
graph.add_node("responder", responder)
graph.add_node("critic", critic)
graph.add_node("improve", improve)
graph.add_node("finalize", finalize)

graph.set_entry_point("detector")

graph.add_edge("detector", "planner")
graph.add_edge("planner", "responder")
graph.add_edge("responder", "critic")

def review_route(state: AssistantState):
    if state.get("critic_count", 0) >= 5:
        return "finalize"
    if state["decision"] == "revise":
        return "improve"
    return "finalize"

graph.add_conditional_edges(
    "critic",
    review_route,
    {
        "improve": "improve",
        "finalize": "finalize"
    }
)

graph.add_edge("improve", "critic")
graph.add_edge("finalize", END)

app = graph.compile()

In [ ]:
app

In [ ]:
result = app.invoke(
    {
        "message": "آیا اگه یخچالم لامپش خاموش باشه غذا خراب میشه؟",
        "critic_count": 0
    }
)

print(result["final_answer"])


In [ ]:
result

In [ ]:
# MasoudKaviani.ir